In [2]:
from collections import Counter, defaultdict

# -------------------------------------------------
# 1. First 20 amino acids of the UNIPROT SEQUENCE P04637|P53_HUMAN Cellular tumor antigen p53
# -------------------------------------------------
seq = "MEEPQSDPSVEPPLSQETFSD"

aa_mass = {
    "G":57,"A":71,"S":87,"P":97,"V":99,"T":101,"C":103,
    "I":113,"L":113,"N":114,"D":115,"Q":128,"K":128,
    "E":129,"M":131,"H":137,"F":147,"R":156,"Y":163,"W":186
}

true_peptide = [aa_mass[a] for a in seq]

# -------------------------------------------------
# 2. Linear spectrum from true peptide
# -------------------------------------------------
def linear_spectrum(peptide):
    prefix = [0]
    for m in peptide:
        prefix.append(prefix[-1] + m)

    spec = [0]
    n = len(peptide)
    for i in range(n):
        for j in range(i+1, n+1):
            spec.append(prefix[j] - prefix[i])
    return sorted(spec)

Spectrum = linear_spectrum(true_peptide)
parent_mass = max(Spectrum)
spectrum_counter = Counter(Spectrum)

# -------------------------------------------------
# 3. Amino acid masses (18)
# -------------------------------------------------
AA_MASS = sorted(set([
    57, 71, 87, 97, 99, 101, 103, 113,
    114, 115, 128, 129, 131, 137, 147,
    156, 163, 186
]))

mass_to_aa = {
    57:"G",71:"A",87:"S",97:"P",99:"V",101:"T",103:"C",
    113:"I/L",114:"N",115:"D",128:"K/Q",129:"E",131:"M",
    137:"H",147:"F",156:"R",163:"Y",186:"W"
}

# -------------------------------------------------
# 4. Branch & Bound helpers
# -------------------------------------------------
def peptide_mass(p):
    return sum(p)

def is_consistent(peptide):
    lin = linear_spectrum(peptide)
    lin_counter = Counter(lin)
    for m,c in lin_counter.items():
        if spectrum_counter[m] < c:
            return False
    return True

def expand(peptides):
    return [p+[m] for p in peptides for m in AA_MASS]

def peptide_to_mass(p):
    return "-".join(map(str,p))

def peptide_to_aa(p):
    return "-".join(mass_to_aa[m] for m in p)

# -------------------------------------------------
# 5. Branch & Bound
# -------------------------------------------------
peptides = [[]]
consistent_by_len = defaultdict(list)

while peptides:
    peptides = expand(peptides)
    new_list = []

    for pep in peptides:
        mass = peptide_mass(pep)

        if mass > parent_mass:
            continue

        if not is_consistent(pep):
            continue

        new_list.append(pep)
        consistent_by_len[len(pep)].append(pep)

    peptides = new_list

# -------------------------------------------------
# 6. Output
# -------------------------------------------------
print("True peptide:", seq)
print("True masses :", true_peptide)
print("Parent mass:", parent_mass)

max_len = max(consistent_by_len)

for L in range(1, max_len+1):
    print(f"\nConsistent {L}-mers (count = {len(consistent_by_len[L])}):")
    for pep in consistent_by_len[L]:
        print(f"{peptide_to_mass(pep):<30} {peptide_to_aa(pep)}")


True peptide: MEEPQSDPSVEPPLSQETFSD
True masses : [131, 129, 129, 97, 128, 87, 115, 97, 87, 99, 129, 97, 97, 113, 87, 128, 129, 101, 147, 87, 115]
Parent mass: 2329

Consistent 1-mers (count = 11):
87                             S
97                             P
99                             V
101                            T
113                            I/L
115                            D
128                            K/Q
129                            E
131                            M
147                            F
186                            W

Consistent 2-mers (count = 51):
87-97                          S-P
87-99                          S-V
87-113                         S-I/L
87-115                         S-D
87-128                         S-K/Q
87-147                         S-F
97-87                          P-S
97-97                          P-P
97-113                         P-I/L
97-115                         P-D
97-128                         P-K/Q
97-129   